# Data-parallel SGD step with psum

In data parallelism every device holds the full model and a slice of the batch. Each computes local gradients, an all-reduce (`jax.lax.psum`) sums them across devices, and every replica applies the same update. You only have one CPU device, but `jax.vmap(..., axis_name='devices')` runs collectives over a mapped axis exactly as `shard_map` would over real devices.

Model: linear regression, `params = {'w': (D,), 'b': ()}`. Device $n$ holds `x[n]` `(b, D)`, `y[n]` `(b,)` and `mask[n]` `(b,)` of 0/1 (padding examples are 0). The global loss is the mean squared error over all **real** examples on all devices:

$$L = \frac{\sum_{n,i} \text{mask}_{n,i} (x_{n,i} \cdot w + b - y_{n,i})^2}{\max\bigl(1, \sum_{n,i} \text{mask}_{n,i}\bigr)}$$

Write `dp_step(params, x, y, mask, lr)`:

1. A per-device function computes the **local** summed loss, its gradient and the local count of real examples.
2. `psum` the gradient (a pytree), the summed loss and the count over the axis `'devices'`, divide, and apply $p \leftarrow p - \eta \nabla L$.
3. Run it with `jax.vmap(per_device, in_axes=(None, 0, 0, 0, None), axis_name='devices')`.

Return `(new_params, loss)` as the vmap produces them: `new_params` has leaves with a leading device axis (`'w'`: `(N, D)`, `'b'`: `(N,)`), and every row must be identical; `loss` is `(N,)`, also identical across devices. float32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def dp_step(params, x, y, mask, lr):
    """x (N, b, D), y/mask (N, b). Return (per-device new params, per-device loss (N,))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[[0.0]], [[0.0]]], "dtype": "float64"}, {"__nd__": [[2.0], [4.0]], "dtype": "float64"}, {"__nd__": [[1.0], [1.0]], "dtype": "float64"}, 0.25], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [1.5, 1.5], "dtype": "float32"}], ["w", {"__nd__": [[0.0], [0.0]], "dtype": "float32"}]]}, {"__nd__": [10.0, 10.0], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[[0.0], [0.0]], [[0.0], [0.0]]], "dtype": "float64"}, {"__nd__": [[2.0, 2.0], [8.0, 0.0]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0], [1.0, 0.0]], "dtype": "float64"}, 0.25], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [2.0, 2.0], "dtype": "float32"}], ["w", {"__nd__": [[0.0], [0.0]], "dtype": "float32"}]]}, {"__nd__": [24.0, 24.0], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.039924658947135025, -1.2577367209219155, 2.574023189963745], "dtype": "float64"}], ["b", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[0.48179797619192904, 0.6435466245355551, -0.2079266799144402], [0.058287084621405076, 0.3367426363654663, 0.19510708987712772], [-0.6093144861761308, 0.6146803888149932, 0.4904031176447712], [-0.11826438854963027, -0.7192432635329281, -0.8793663661900869], [-1.0080283314520588, 0.2766290585896302, -0.3509065477161458]], [[0.8710605645749822, -0.8877781787569496, 0.7257502459207419], [0.026903576187543587, -0.5688802021416602, 1.0445611423368582], [0.1506798003717234, 0.7902301062665426, 1.091058610815693], [0.49449582356343735, -0.48722566371162235, 0.18579545278185208], [1.0044718838250786, -0.6234189278456499, 0.5457201105900348]], [[-0.1956920931749041, -0.4817714748626647, -0.5617920153636436], [-1.0812397284053499, -0.2647569600243273, 0.3789485891624878], [-1.9769498622123847, -0.39996507030009887, 0.20732724345117484], [-0.39768028018528206, 0.7373163550325441, -0.8107713514355345], [0.6996726552099481, 0.7910266266214417, 0.3744508883837694]], [[-0.7512934307913205, 1.7357286050813117, -0.26527436414612615], [0.4876343772258879, -0.988214645210646, -0.7221759973025274], [0.3632501320814856, -1.400600774772154, 0.4177497310474027], [1.7087803889003808, -1.429500504021974, -0.3130516159073398], [0.8128742268098668, -0.37711280575343364, 0.7859829678314249]]], "dtype": "float64"}, {"__nd__": [[0.13949074051629437, -1.6979212108333603, -2.600474603677622, -0.2893043919490064, 0.868000059098014], [-0.8806672032782616, 1.1804593028304717, -1.625208580989776, 0.5999974787511552, -0.23273173844029318], [-0.16006599031150895, -0.39834651858816056, -0.7028792098032082, -0.13438669052043656, -1.1462801494511399], [0.8697187279829661, -1.2075123582178628, 0.5521211187371025, -1.0957431543778011, -1.3599776574796472]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [-0.10681616514921188, -0.10681616514921188, -0.10681616514921188, -0.10681616514921188], "dtype": "float32"}], ["w", {"__nd__": [[-0.1732889711856842, -1.0694737434387207, 2.350426435470581], [-0.1732889711856842, -1.0694737434387207, 2.350426435470581], [-0.1732889711856842, -1.0694737434387207, 2.350426435470581], [-0.1732889711856842, -1.0694737434387207, 2.350426435470581]], "dtype": "float32"}]]}, {"__nd__": [5.796156883239746, 5.796156883239746, 5.796156883239746, 5.796156883239746], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.5874726488138801, 0.31969520068603396, 0.03768513541428808], "dtype": "float64"}], ["b", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[1.7371297821515936, -0.7792030354898237, 0.9116042242001557], [0.3108029666553994, 1.6977407306754846, -1.0841930457565077], [-0.3763791438158808, -0.5358175519335485, -0.8913813179772303], [-1.519268316029521, -0.25310909998157294, -0.30267668163023453], [-0.9363531558378526, -0.2589173667480098, -1.7343381007258722]], [[1.3847343557741991, 0.478468620469388, 0.17546013143733777], [0.6878982529467101, -0.36866455697741696, -0.6755069374160801], [-0.5768738090978073, -0.7755081122661167, -0.47254359690726455], [1.028021246430097, -0.5086072390834352, 1.3579819009966394], [0.3944881300514539, 1.8699447435302585, -0.5391774039359204]], [[-0.7310781834937583, -0.6420947009276423, -2.1808324015899063], [0.7267053512997421, 0.43732957954775814, 0.10692979104346301], [0.08247960328328352, 0.9244051757530275, -0.9005314345659461], [0.3377210691172636, 0.3528613400217019, 0.7745718487935188], [1.4624855101434717, 0.4686933272268092, -0.7716124219912571]], [[0.5353969615861389, -0.2550202737400714, 0.07815211279734788], [-1.0085380963193316, -0.2755499530682979, -0.04576293363102138], [-0.28283760324467594, 0.5085877835976157, 1.8325747950810989], [-0.024093818057985265, -0.8725681019979795, -0.2780823511833418], [1.1358369114760958, 0.099928827232368, 1.0136611351379152]]], "dtype": "float64"}, {"__nd__": [[0.21821014536931185, 1.0061786432256317, -0.3907024939181706, 1.3235417076525016, 0.2601674060027494], [0.6236809984751422, 1.4224679690236088, -2.1436874881221684, 1.4171578068241923, 2.440057581199923], [-0.1798145030166986, -0.5121815856871224, -0.15633348075828316, 0.7033630712486818, -0.08553999901719583], [-1.1903984013451647, -0.9978495859024343, -1.6728281029737238, -0.7208322060743139, -0.9763343711226508]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 0.0, 1.0, 0.0], [1.0, 1.0, 1.0, 0.0, 1.0], [0.0, 0.0, 1.0, 1.0, 0.0], [0.0, 0.0, 1.0, 1.0, 1.0]], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [0.06289712339639664, 0.06289712339639664, 0.06289712339639664, 0.06289712339639664], "dtype": "float32"}], ["w", {"__nd__": [[0.515659511089325, 0.36610737442970276, -0.07451917231082916], [0.515659511089325, 0.36610737442970276, -0.07451917231082916], [0.515659511089325, 0.36610737442970276, -0.07451917231082916], [0.515659511089325, 0.36610737442970276, -0.07451917231082916]], "dtype": "float32"}]]}, {"__nd__": [1.5903719663619995, 1.5903719663619995, 1.5903719663619995, 1.5903719663619995], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.7642808356493177, -2.1549067087414424], "dtype": "float64"}], ["b", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[0.2781294573480191, -0.14097560508913837], [0.7640264340493589, 1.2110845764566323]], [[0.4812196640222154, 0.15646777768921588], [-0.7061183266453435, 0.9732974171534232]], [[-0.00800477280325207, 0.32765542752605537], [-0.15653190784085608, 0.11497000595693163]]], "dtype": "float64"}, {"__nd__": [[-0.7141177164011101, 2.525315739224063], [0.15750583299200863, -0.42690481510505496], [-0.45527157198854157, 0.3310177791501134]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [1.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [0.1268923282623291, 0.1268923282623291, 0.1268923282623291], "dtype": "float32"}], ["w", {"__nd__": [[0.7772219777107239, -2.1506989002227783], [0.7772219777107239, -2.1506989002227783], [0.7772219777107239, -2.1506989002227783]], "dtype": "float32"}]]}, {"__nd__": [0.0007231976487673819, 0.0007231976487673819, 0.0007231976487673819], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.46685523555965597, 0.7375384373248398], "dtype": "float64"}], ["b", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[0.03542768542376541, 0.27591796630776405], [1.1110286181697488, 0.32175388248358555], [0.40841234982229474, 0.5528055648070976]], [[1.3850118426526896, -0.08581069847001192], [1.2280695765975334, 0.749390621837444], [0.4969971992808709, -1.2138629555767348]]], "dtype": "float64"}, {"__nd__": [[0.18817734635125408, -1.3853070441195225, -0.5991021340490555], [-0.28490360961627664, -0.5030531669494375, -1.3503287443886873]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [0.10000000149011612, 0.10000000149011612], "dtype": "float32"}], ["w", {"__nd__": [[-0.4668552279472351, 0.7375384569168091], [-0.4668552279472351, 0.7375384569168091]], "dtype": "float32"}]]}, {"__nd__": [0.0, 0.0], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.9005377779726241, -0.23803059470179652, 1.3151645904541476, -0.0186947318897892, -1.8305924606600128, 1.395066221536367], "dtype": "float64"}], ["b", {"__nd__": 0.1, "dtype": "float64"}]]}, {"__nd__": [[[0.9328855882739421, 1.6481899089195127, -0.35090582171245427, -2.1033122489650187, 0.09989526256811314, -1.1244740291744433], [-0.9347522218878793, -1.1435279589175043, -1.263192466601223, 0.9631084449812484, -1.1729140951968295, -2.344872548779239], [1.7058328764138986, -0.8753100968342761, -1.2365650633192893, 0.7824072121753584, 0.16926576105873906, -1.736611446965252], [0.8863786020299813, 0.7592287411197484, 0.30482280547861224, -1.8976198898380363, -0.10340808724790457, -1.9749141593210766]], [[0.10282020962553926, 0.5054076066330351, 0.7817034991595999, -0.9566864941935743, 1.3244020263239593, -0.250679681746993], [-0.3856721543115872, -0.8410995181898656, 0.38497417044020255, 0.51506080032673, -0.12869904058057188, -0.8150324430341086], [1.2634705750151707, 0.20956589496649156, 0.7943045907385781, 0.5329652995677678, 0.08053638390814688, -0.7193095157604033], [-0.6269447986377443, -0.3993061064906921, -2.569365077508924, -0.7098703851748295, 0.8514281221533981, 0.03863136054418396]], [[-1.4761243355903637, 0.36909311537375966, -0.46601947887087786, 0.9983820623073227, -0.27875158840808023, -0.9142008590107317], [0.08692282164005878, -0.35391516413848445, 0.5668978889213512, -0.6610390005525854, 1.198450249526036, -0.32671124117337946], [-1.3122138105814694, -0.6573523889027931, -0.458526696660812, 0.8465969761320612, 0.3960284301834315, -0.07625118656308795], [0.8291938493517195, 0.35666372034287974, 0.9186010956883651, -0.39080797162072406, 1.4522129681018066, -1.9427694531690154]], [[-1.3347655533149734, 0.017923534151238565, 0.7879365952458153, 1.4407695587642284, 0.23842750562456916, 0.08612004398966724], [0.38798196361938875, 0.24318129887330356, -0.011750971487027613, -1.2164309808975051, -0.8153457228254777, -0.02783935372382832], [-1.4773942488658278, -0.07060528712772979, 0.9716749670941301, -0.3631702886848186, -1.052483452294115, -1.4633423480338086], [-0.8072975109767416, 0.05813330789311603, 0.8185924749852367, -0.14443594535962814, 0.39613380440139834, 0.48756641417835594]], [[-0.2855380856505027, -0.7725434110952345, 1.3483966609827522, 0.5674996571928507, 1.3277515046797423, -0.5222159242791536], [-0.39000623276605084, 0.1747228499462566, -0.460318251553863, -1.2137635269617393, -0.7747988773697458, -1.310884774371793], [-0.3273951717386142, -2.064184289332859, 0.42872075471721544, 0.6157761743689726, -1.99011894492324, -0.5263262180318745], [-0.07327227425663173, -1.293671171809337, -0.1122229546596502, 0.33880954887163606, -0.6682826451106892, 0.7473422900198942]], [[0.04763071416797451, -1.2876654930029954, -0.9701640195803781, 1.1837993386392194, 1.3185067752864768, 0.19658883674515315], [1.2775611793158403, -1.3632552415557173, 1.621605482120588, 1.671512167844944, -0.32450987080209726, -0.1501546658909833], [-1.0893411911174473, 2.827931674710007, -0.002713043259787372, -0.29992909885748875, -0.8006389891567881, -0.573031419254063], [-0.5828963262058808, 0.978503468437502, 0.6016019231067702, -0.8063419803061401, 0.16678029800542915, -0.7954858137000019]], [[0.13481344570317427, -2.7799772919008947, 1.7473870286681341, 0.31397877732416796, -0.8934306875533052, 0.950580110064372], [0.09159414920653522, -0.32677460524102064, 1.738556739600574, 0.31811671005220354, 0.7398652702970896, 1.326967315979171], [0.4357756732907719, -0.008847025389778376, -0.14694732017370993, -1.056011426039622, 0.7890430137853287, -1.7896172860756605], [0.7409221040071483, 1.0702999276576592, 0.6818244029302104, 0.008650765460492996, -0.830061480312499, 0.4290572071509295]], [[1.285508789933764, 0.6301729995242236, 0.8892713381872239, -0.6823247173584706, -0.5818448914470378, 0.5428007373204071], [-2.016505576701858, -0.6857854779367999, -1.5329508412025374, -1.6837288049343755, 0.59388816940022, 0.911551153878885], [0.1682994284706973, -0.3088070171594107, -1.0802164542170027, -0.026913021818901282, -0.7187869223030238, 0.8509470509936765], [-1.4795372780880307, -0.5919636215867238, 1.432363266941302, -1.07427495096555, 0.5603054206630944, -0.42900791374908664]]], "dtype": "float64"}, {"__nd__": [[-15.920601775165625, 0.4911209835016908, 0.1340908050651145, -6.702672138696917], [1.4207420322993902, 5.818630435319463, -0.9207947731388038, 2.0618360176911725], [-4.5819091763459845, 6.976799911871679, -0.47343340333496964, 0.49448264916848106], [-6.1012263815406245, -2.8858358875608587, 5.577979086086496, 1.8553719993963405], [-4.432547109893555, -9.064005391676213, -10.323791852928268, 3.0206327166845943], [-5.036131634893057, -1.2921444431556404, 8.3077828783252, -6.582301748229149], [-3.8711519692314003, 5.320162954962866, 0.6674170768643064, 5.371080936560524], [5.2446921199871746, -3.14481070998965, -2.2339633320338903, 4.131885059863024]], "dtype": "float64"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]], "dtype": "float64"}, 0.01], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": [0.08990666270256042, 0.08990666270256042, 0.08990666270256042, 0.08990666270256042, 0.08990666270256042, 0.08990666270256042, 0.08990666270256042, 0.08990666270256042], "dtype": "float32"}], ["w", {"__nd__": [[0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122], [0.8781185150146484, -0.2121405452489853, 1.2981359958648682, -0.01630375161767006, -1.795211911201477, 1.3909693956375122]], "dtype": "float32"}]]}, {"__nd__": [35.14942169189453, 35.14942169189453, 35.14942169189453, 35.14942169189453, 35.14942169189453, 35.14942169189453, 35.14942169189453, 35.14942169189453], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(dp_step, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")